In [1]:
import numpy as np
from tensorflow.keras.applications import ResNet50, MobileNetV2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, UpSampling2D, concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# Build Classification Model
def build_classification_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    model = Sequential([
        base_model,
        Flatten(),
        Dense(128, activation='relu'),
        Dense(2, activation='softmax')  # Binary classification
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model



In [2]:
# Build Segmentation Model
def build_segmentation_model(input_shape=(224, 224, 3)):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=input_shape)
    skip_connections = [base_model.get_layer(name).output for name in ['block_1_expand_relu', 'block_3_expand_relu', 'block_6_expand_relu', 'block_13_expand_relu']]
    x = base_model.output
    for skip in reversed(skip_connections):
        x = UpSampling2D((2, 2))(x)
        x = concatenate([x, skip])
        x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = UpSampling2D((2, 2))(x)
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(x)
    model = Model(base_model.input, outputs)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model



In [5]:
# Train Classification Model
classification_model = build_classification_model()
train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_generator = train_datagen.flow_from_directory(
    'C:/Users/verti/Documents/spine/newdataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)
val_generator = train_datagen.flow_from_directory(
    'C:/Users/verti/Documents/spine/newdataset/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 688 images belonging to 2 classes.
Found 48 images belonging to 2 classes.


In [6]:
classification_model.fit(train_generator, validation_data=val_generator, epochs=10)
classification_model.save_weights('classification_model.h5')

# Train Segmentation Model
segmentation_model = build_segmentation_model()
# Assuming 'train_images' and 'train_masks' are numpy arrays of images and masks
segmentation_model.fit(train_images, train_masks, validation_split=0.2, epochs=10)
segmentation_model.save_weights('segmentation_model.h5')


C:\Users\verti\anaconda3\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 164s 6s/step - accuracy: 0.4936 - loss: 18.4911 - val_accuracy: 0.5625 - val_loss: 0.6930
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 145s 7s/step - accuracy: 0.5907 - loss: 0.6820 - val_accuracy: 0.5625 - val_loss: 0.6923
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 157s 7s/step - accuracy: 0.6014 - loss: 0.6652 - val_accuracy: 0.5625 - val_loss: 0.6915
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 143s 7s/step - accuracy: 0.6268 - loss: 0.6447 - val_accuracy: 0.5625 - val_loss: 0.6910
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 146s 7s/step - accuracy: 0.6438 - loss: 0.6360 - val_accuracy: 0.5625 - val_loss: 0.6907
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 147s 7s/step - accuracy: 0.6622 - loss: 0.5843 - val_accuracy: 0.5625 - val_loss: 1.4766
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 169s 8s/step - accuracy: 0.6752 - loss: 0.5325 - val_accuracy: 0.5625 - val_loss: 1.3340
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 165s 7s/step - accuracy: 0.7284 - loss: 0.4843 - val_accuracy: 0.5625 - 

ValueError: The filename must end in `.weights.h5`. Received: filepath=classification_model_weights.h5

#### 

In [12]:
# import cv2
# import numpy as np
# import os

# def generate_synthetic_masks(image_dir, output_dir):
#     os.makedirs(output_dir, exist_ok=True)
#     for filename in os.listdir(image_dir):
#         if filename.endswith('.jpg') or filename.endswith('.png'):
#             image_path = os.path.join(image_dir, filename)
#             image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
#             _, mask = cv2.threshold(image, 127, 255, cv2.THRESH_BINARY)
#             output_path = os.path.join(output_dir, filename)
#             cv2.imwrite(output_path, mask)

# # Example usage:
# image_dir = 'C:/Users/verti/Documents/spine/orginal_data/c_data/normal'
# output_dir = 'C:/Users/verti/Documents/spine/orginal_data/m_data/normal'
# generate_synthetic_masks(image_dir, output_dir)


In [16]:
# # pip install split-folders[full]

# #### Import the package

# import splitfolders
# ###### Path of input folder

# input_folder = 'C:/Users/verti/Documents/spine/orginal_data/m_data'
# #######ration of split is 70%, 20% and 10%
# splitfolders.ratio(input_folder, output="C:/Users/verti/Documents/spine/orginal_data/mask_data", 
#                    seed=42, ratio=(.7, .3,), 
#                    group_prefix=None)

In [17]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import Xception
from tensorflow.keras.models import Sequential, Model, load_model
from tensorflow.keras.layers import Dense, Flatten, Dropout, Conv2D, MaxPooling2D, UpSampling2D, Input, concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import os

# 1. Build Classification Model
def build_classification_model():
    base_model = Xception(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    model = Sequential([
        base_model,
        Flatten(),
        Dense(256, activation='relu'),
        Dropout(0.5),  # Regularization
        Dense(128, activation='relu'),
        Dense(2, activation='softmax')  # Binary classification
    ])
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


In [18]:

# 2. Build U-Net Segmentation Model
def build_unet_model(input_shape=(224, 224, 3)):
    inputs = Input(input_shape)
    
    # Encoder
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)
    
    # Bottleneck
    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    
    # Decoder
    u1 = UpSampling2D((2, 2))(c3)
    u1 = concatenate([u1, c2])
    c4 = Conv2D(128, (3, 3), activation='relu', padding='same')(u1)
    c4 = Conv2D(128, (3, 3), activation='relu', padding='same')(c4)
    
    u2 = UpSampling2D((2, 2))(c4)
    u2 = concatenate([u2, c1])
    c5 = Conv2D(64, (3, 3), activation='relu', padding='same')(u2)
    c5 = Conv2D(64, (3, 3), activation='relu', padding='same')(c5)
    
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    
    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 3. Create data generators for classification and segmentation
def create_classification_generators(train_dir, val_dir):
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True
    )
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical'
    )
    
    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical'
    )
    return train_generator, val_generator

def create_segmentation_generators(train_images_dir, train_masks_dir, batch_size=32):
    train_image_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
        train_images_dir, target_size=(224, 224), batch_size=batch_size, class_mode=None
    )
    train_mask_generator = ImageDataGenerator(rescale=1./255).flow_from_directory(
        train_masks_dir, target_size=(224, 224), batch_size=batch_size, class_mode=None, color_mode='grayscale'
    )
    while True:
        x = train_image_generator.next()
        y = train_mask_generator.next()
        yield x, y



In [19]:
# 4. Train Classification Model
def train_classification_model(model, train_generator, val_generator):
    model.fit(
        train_generator,
        epochs=20,
        validation_data=val_generator,
        steps_per_epoch=len(train_generator),
        validation_steps=len(val_generator)
    )
    model.save_weights('xception_classification_weights.h5')

# 5. Train Segmentation Model
def train_segmentation_model(model, train_generator):
    model.fit(train_generator, epochs=20, steps_per_epoch=100)
    model.save_weights('unet_segmentation_weights.h5')

# 6. Perform classification and segmentation
def classify_and_segment(image_path, classification_model, segmentation_model):
    # Classification
    image = load_img(image_path, target_size=(224, 224))
    image_array = img_to_array(image) / 255.0
    image_array = np.expand_dims(image_array, axis=0)
    class_prediction = classification_model.predict(image_array)
    class_labels = ['Normal', 'Abnormal']
    predicted_class = class_labels[np.argmax(class_prediction)]
    
    print(f"Predicted Class: {predicted_class}")
    plt.imshow(image)
    plt.title(f"Classification: {predicted_class}")
    plt.axis('off')
    plt.show()
    
    # Segmentation (if abnormal)
    if predicted_class == 'Abnormal':
        segmentation_prediction = segmentation_model.predict(image_array)
        segmentation_mask = segmentation_prediction.squeeze()
        plt.imshow(segmentation_mask, cmap='gray')
        plt.title('Segmentation Mask')
        plt.axis('off')
        plt.show()



In [21]:
# Main Workflow
if __name__ == "__main__":
    # Paths to datasets
    train_class_dir = 'C:/Users/verti/Documents/spine/orginal_data/classification_data/train'
    val_class_dir = 'C:/Users/verti/Documents/spine/orginal_data/classification_data/val'
    train_seg_images_dir = 'C:/Users/verti/Documents/spine/orginal_data/classification_data/train'
    train_seg_masks_dir = 'C:/Users/verti/Documents/spine/orginal_data/mask_data/train'
    
    # Build models
    classification_model = build_classification_model()
    segmentation_model = build_unet_model()
    
    # Create data generators
    class_train_gen, class_val_gen = create_classification_generators(train_class_dir, val_class_dir)
    seg_train_gen = create_segmentation_generators(train_seg_images_dir, train_seg_masks_dir)
    
    # Train models
    print("Training Classification Model...")
    train_classification_model(classification_model, class_train_gen, class_val_gen)
    
    print("Training Segmentation Model...")
    train_segmentation_model(segmentation_model, seg_train_gen)
    
    # Perform classification and segmentation
    test_image_path = 'path_to_test_image.jpg'
    classification_model.load_weights('xception_classification_weights.h5')
    segmentation_model.load_weights('unet_segmentation_weights.h5')
    classify_and_segment(test_image_path, classification_model, segmentation_model)


Found 860 images belonging to 2 classes.
Found 369 images belonging to 2 classes.
Training Classification Model...


C:\Users\verti\anaconda3\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 36s/step - accuracy: 0.5472 - loss: 0.7574 

C:\Users\verti\anaconda3\lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


27/27 ━━━━━━━━━━━━━━━━━━━━ 1036s 37s/step - accuracy: 0.5470 - loss: 0.7562 - val_accuracy: 0.5637 - val_loss: 0.7027
Epoch 2/20


C:\Users\verti\anaconda3\lib\contextlib.py:137: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 29ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 3/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1136s 42s/step - accuracy: 0.5542 - loss: 0.6899 - val_accuracy: 0.5664 - val_loss: 0.6894
Epoch 4/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 5/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1212s 45s/step - accuracy: 0.5443 - loss: 0.6911 - val_accuracy: 0.5664 - val_loss: 0.6903
Epoch 6/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 7/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1118s 40s/step - accuracy: 0.5347 - loss: 0.6942 - val_accuracy: 0.5637 - val_loss: 0.6907
Epoch 8/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 9/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 933s 35s/step - accuracy: 0.5502 - loss: 0.6911 - val_accuracy: 0.5637 - val_loss: 0.6904
Epoch 10/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.0000e+00 - loss: 0.0000e+00
Epoch 11/20
27/27 ━━━━━━━━━━━━

ValueError: The filename must end in `.weights.h5`. Received: filepath=xception_classification_weights.h5